In [51]:
import pandas as pd
import pickle
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# 1. 데이터 로드 및 합치기
analysis_data = pd.read_csv('/home/user2/~/A-Awake/src/Python/Data/output_analysis.csv')
analysis_data['target'] = 0  # 정상 파일은 '0'으로 레이블 지정
mal_data = pd.read_csv('/home/user2/~/A-Awake/src/Python/Data/output_mal.csv')
mal_data['target'] = 1  # 악성 파일은 '1'로 레이블 지정
data = pd.concat([analysis_data, mal_data], ignore_index=True)

# 2. 특징(features)과 레이블(labels) 분리
features = data.drop('target', axis=1)
labels = data['target']

# 3. 숫자형 데이터만 선택 (문자열 열 제거)
features_numeric = features.select_dtypes(include=[np.number])

# 4. 데이터셋 분할 (훈련 및 테스트)
X_train, X_test, y_train, y_test = train_test_split(features_numeric, labels, test_size=0.2, random_state=42)

# 5. 결측치 처리 (평균으로 대체)
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

'''
# 6. 스케일링 (특징 스케일 표준화)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)
'''

# 7. 상수 피처 제거
constant_filter = VarianceThreshold(threshold=0)  # threshold=0: 상수 피처 제거
X_train_filtered = constant_filter.fit_transform(X_train_scaled)
X_test_filtered = constant_filter.transform(X_test_scaled)


# 8. 특징 선택 (상위 10개 특징 선택)
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train_filtered, y_train)
X_test_selected = selector.transform(X_test_filtered)

# 선택된 피처 목록 저장
selected_features = [features_numeric.columns[i] for i in selector.get_support(indices=True)]
    
# 9. 하이퍼파라미터 튜닝 및 모델 정의
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10, min_samples_split=10, min_samples_leaf=5)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, min_samples_split=10, min_samples_leaf=5)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3, min_samples_split=10, min_samples_leaf=5)

dt_model.fit(X_train_selected, y_train)
rf_model.fit(X_train_selected, y_train)
gb_model.fit(X_train_selected, y_train)

# 10. 앙상블 모델 정의 (Voting Classifier)
ensemble_model = VotingClassifier(
    estimators=[
        ('decision_tree', dt_model),
        ('random_forest', rf_model),
        ('gradient_boosting', gb_model)
    ],
    voting='soft'  # 'hard'는 다수결 투표, 'soft'는 확률값의 평균으로 결정
)

# 11. 교차 검증을 통한 모델 성능 확인
cv_scores = cross_val_score(ensemble_model, X_train_selected, y_train, cv=5)
print("\nCross-validation scores:", cv_scores)
print("Mean CV Score:", np.mean(cv_scores))

# 12. 앙상블 모델 훈련
ensemble_model.fit(X_train_selected, y_train)

# 13. 예측 수행
ensemble_predictions = ensemble_model.predict(X_test_selected)

# 14. 모델 평가 및 결과 출력
print("\nEnsemble Model Accuracy:", accuracy_score(y_test, ensemble_predictions))
print("\nEnsemble Classification Report:\n", classification_report(y_test, ensemble_predictions))
print("\nEnsemble Confusion Matrix:\n", confusion_matrix(y_test, ensemble_predictions))


Cross-validation scores: [1.         1.         1.         1.         0.99678457]
Mean CV Score: 0.9993569131832798

Ensemble Model Accuracy: 1.0

Ensemble Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       198
           1       1.00      1.00      1.00       191

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389


Ensemble Confusion Matrix:
 [[198   0]
 [  0 191]]


In [52]:
import pickle

# 모델 학습
ensemble_model.fit(X_train_selected, y_train)


# 모델 저장 
model_filename = '/home/user2/~/A-Awake/src/Python/Model/docx_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(ensemble_model, file)

In [53]:
selected_features

['file_size',
 'modification_time',
 'access_time',
 'creation_time',
 'extension_mime_match',
 'num_paragraphs',
 'word_count',
 'char_count_without_spaces',
 'image_count',
 'has_modified_date']

In [50]:
# 모델 학습 시, 전체 피처 목록 저장
all_features = list(features_numeric.columns)  # 전체 피처 목록 추출



In [10]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. 새로운 데이터 로드
new_data_path = "/home/user2/~/A-Awake/src/Python/Data/output_docx_merge.csv"  # 새 데이터 파일의 경로를 입력
new_data = pd.read_csv(new_data_path)

with open(model_filename, 'rb') as file:
    loaded_model = pickle.load(file)

# 2. 기존 사용 피처 동일하게 유지
# 이전에 사용된 피처 리스트 (used_features)
print("Original used features:", used_features)

# 3. 새로운 데이터에서 동일한 피처 선택
new_features = new_data[used_features]

# 4. 숫자형 데이터만 선택 (문자열 열 제거)
new_features_numeric = new_features.select_dtypes(include=[np.number])

# 5. 결측치 처리 (평균으로 대체)
imputer = SimpleImputer(strategy='mean')
new_features_imputed = imputer.fit_transform(new_features_numeric)

# 6. 스케일링 (기존 훈련 데이터의 스케일러 사용)
new_features_scaled = scaler.transform(new_features_imputed)

# 7. 상수 피처 제거 (기존 훈련된 constant_filter 사용)
new_features_filtered = constant_filter.transform(new_features_scaled)

# 8. 특징 선택 (기존 훈련된 selector 사용)
new_features_selected = selector.transform(new_features_filtered)

# 9. 예측 수행 (앙상블 모델 사용)
new_predictions = ensemble_model.predict(new_features_selected)

# 10. 예측 결과 요약
print("\nPredictions on new data:")
print("Malicious (1):", sum(new_predictions == 1))
print("Normal (0):", sum(new_predictions == 0))

# 11. 결과 데이터프레임으로 보기 (Optional)
new_data['Prediction'] = new_predictions
print("\nNew data with predictions:")
print(new_data[['Prediction']].value_counts())

Original used features: ['file_size', 'modification_time', 'access_time', 'creation_time', 'mime_type_is_standard', 'extension_mime_match', 'num_paragraphs', 'word_count', 'char_count_with_spaces', 'char_count_without_spaces', 'avg_word_length', 'avg_paragraph_length', 'hyperlink_count', 'num_tables', 'image_count', 'unique_styles', 'embedded_object_count', 'has_author', 'has_last_modified_by', 'has_creation_date', 'has_modified_date', 'header_length', 'footer_length', 'num_sections', 'special_char_count', 'numeric_char_ratio', 'uppercase_char_ratio', 'keyword_invoice', 'keyword_urgent', 'keyword_payment', 'keyword_account', 'keyword_click here', 'keyword_update', 'keyword_verify', 'keyword_password', 'unique_fonts', 'is_encrypted', 'ole_object_count', 'has_macros', 'unique_image_formats', 'num_page_orientations', 'num_page_sizes', 'comments_count', 'revisions_count', 'field_codes_count', 'form_fields_count', 'bookmarks_count', 'unique_hyperlink_domains', 'media_object_count']

Predict

In [2]:
import pandas as pd

# 2. CSV 파일 읽기 (파일 경로에 맞게 수정)
csv_file_path = r'C:\Users\김진하\Documents\VSCODE\A_DocMal\Mail_Maker\git\A-Awake\src\Python\output_docx_merge.csv'  # 여기에 CSV 파일 경로를 넣으세요
data = pd.read_csv(csv_file_path)

X_sample = data.iloc[3, :].values.reshape(1, -1)
# # 3. 첫 번째 행 데이터 추출 (특정 열을 선택할 수도 있음)
# # 여기서는 첫 번째 행 전체 데이터를 사용한다고 가정
# X_sample = data.iloc[0, :-1].values.reshape(1, -1)  # 마지막 열이 라벨이라 가정하고 제외

In [4]:
X_sample

array([['7754428c05c3bd2a2f43150b5e0b1553924c5e5ef9d939fd456a96fc69657389.docx',
        159506, 1726882825.259756, 1726891562.2106533,
        1727675134.9510612,
        'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
        1, '.docx', 1, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan]], dtype=object)

In [3]:
import joblib
import numpy as np
import sklearn

# 1. 모델 불러오기
model_path = 'ensemble_model.pkl'
loaded_model = joblib.load(model_path)

# 3. 모델로 테스트 데이터 예측
y_pred = loaded_model.predict(X_sample)

# 4. 예측 결과 출력
print("모델이 예측한 결과 (라벨):")
print(y_pred)


c:\anaconda3\envs\mail_Maker\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.2.2 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\anaconda3\envs\mail_Maker\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.2.2 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\anaconda3\envs\mail_Maker\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator GradientBoostingClassifier from version 1.2.2 when using ve

ValueError: node array from the pickle has an incompatible dtype:
- expected: {'names': ['left_child', 'right_child', 'feature', 'threshold', 'impurity', 'n_node_samples', 'weighted_n_node_samples', 'missing_go_to_left'], 'formats': ['<i8', '<i8', '<i8', '<f8', '<f8', '<i8', '<f8', 'u1'], 'offsets': [0, 8, 16, 24, 32, 40, 48, 56], 'itemsize': 64}
- got     : [('left_child', '<i8'), ('right_child', '<i8'), ('feature', '<i8'), ('threshold', '<f8'), ('impurity', '<f8'), ('n_node_samples', '<i8'), ('weighted_n_node_samples', '<f8')]